In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 2-Stage Optuna Optimization & LightGBM vs XGBoost Benchmark (`models/optimize_soft_pipeline_thresholds.ipynb`)

This notebook implements a **2-Stage Optuna Optimization Strategy** (LightGBM Hyperparameters -> Class Multiplier Weights) and compares LightGBM vs XGBoost using the **38-feature uniform input matrix** and **SMOTE upsampling**:

### Log Silence & Optimization Workflow
1. **Silenced Training Logs**: Configured `verbosity = -1L` on LightGBM and `verbose = 0` on XGBoost.
2. **Stage 1 (LightGBM Hyperparameter Tuning)**: 100 Optuna trials tuning LightGBM parameters to maximize validation **Macro Balanced Accuracy**.
3. **Stage 2 (Class Multiplier Weight Tuning)**: 200 Optuna trials tuning class decision weights $\mathbf{w}^* = [w_1, w_2, w_3, w_4, w_5]$.
4. **Baseline Algorithm Comparison**: Compares LightGBM vs XGBoost sub-models and combined soft pipelines on the holdout test set.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Build Uniform 38-Feature Matrix
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
  library(xgboost)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
fit_scaler <- function(df_train, cols) {
  means <- colMeans(df_train[, cols, drop = FALSE], na.rm = TRUE)
  sds   <- apply(df_train[, cols, drop = FALSE], 2, sd, na.rm = TRUE)
  sds[sds == 0] <- 1
  return(list(means = means, sds = sds, cols = cols))
}
apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}
smote_binary_data <- function(df, feat_cols, label_vec, seed = 42) {
  set.seed(seed)
  pos_idx <- which(label_vec == 1)
  neg_idx <- which(label_vec == 0)
  n_pos <- length(pos_idx)
  n_neg <- length(neg_idx)
  if (n_pos == 0 || n_neg == 0 || n_pos == n_neg) return(list(X = as.matrix(df[, feat_cols]), y = label_vec))
  
  if (n_pos < n_neg) {
    minority_idx <- pos_idx
    target_syn   <- n_neg - n_pos
    min_label    <- 1
  } else {
    minority_idx <- neg_idx
    target_syn   <- n_pos - n_neg
    min_label    <- 0
  }
  
  X_min <- as.matrix(df[minority_idx, feat_cols, drop = FALSE])
  syn_matrix <- matrix(0, nrow = target_syn, ncol = length(feat_cols))
  
  for (i in 1:target_syn) {
    base_i <- sample(1:nrow(X_min), 1)
    nn_i   <- sample(1:nrow(X_min), 1)
    alpha  <- runif(1, 0, 1)
    syn_matrix[i, ] <- X_min[base_i, ] + alpha * (X_min[nn_i, ] - X_min[base_i, ])
  }
  
  X_full <- rbind(as.matrix(df[, feat_cols]), syn_matrix)
  y_full <- c(label_vec, rep(min_label, target_syn))
  return(list(X = X_full, y = y_full))
}
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
hr_rng   <- pulse_max - pulse_min
sbp_rng  <- sbp_max - sbp_min
rr_rng   <- resp_max - resp_min
spo2_rng <- spo2_max - spo2_min
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_min               = pulse_min,
  resp_min                = resp_min,
  spo2_min                = spo2_min,
  sbp_min                 = sbp_min,
  pulse_max               = pulse_max,
  resp_max                = resp_max,
  spo2_max                = spo2_max,
  sbp_max                 = sbp_max,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  hr_range                = hr_rng,
  rr_range                = rr_rng,
  spo2_range              = spo2_rng,
  sbp_range               = sbp_rng,
  shock_index             = t_hr / ifelse(t_sbp == 0, 1, t_sbp),
  hr_mid_to_triage        = t_hr - hr_rng,
  sbp_mid_to_triage       = t_sbp - sbp_rng,
  rr_mid_to_triage        = t_rr - rr_rng,
  spo2_mid_to_triage      = t_o2 - spo2_rng,
  rox_index               = t_o2 / ifelse(t_rr == 0, 1, t_rr),
  spo2_drop_ratio         = spo2_rng / ifelse(spo2_max == 0, 1, spo2_max),
  hr_instability_ratio    = hr_rng / (t_hr + 1),
  bif                     = (t_rr / ifelse(t_o2 == 0, 1, t_o2)) * 100
)
layer_feat_names <- names(df_master)
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- stratified_partition(train_val_df$target_col, p = 1 - rel_val_size, seed = config$training$random_state)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
cont_cols <- c("age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2", "pulse_min", "resp_min", "spo2_min", "sbp_min", "pulse_max", "resp_max", "spo2_max", "sbp_max", "hr_range", "rr_range", "spo2_range", "sbp_range", "shock_index", "hr_mid_to_triage", "sbp_mid_to_triage", "rr_mid_to_triage", "spo2_mid_to_triage", "rox_index", "spo2_drop_ratio", "hr_instability_ratio", "bif")
scaler <- fit_scaler(train_df, cont_cols)
train_scaled <- apply_scaler(train_df, scaler)
val_scaled   <- apply_scaler(val_df, scaler)
test_scaled  <- apply_scaler(test_df, scaler)
cat(sprintf("Partitions Prepared with 38 Uniform Features: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
# ---------------------------------------------------------
# Step 2: STAGE 1 OPTUNA - Tune LightGBM Hyperparameters (Silenced Logs)
# ---------------------------------------------------------
import os
import json
import numpy as np
import pandas as pd
import optuna
from rpy2.robjects import r
r('''
evaluate_lgb_hyperparams <- function(lr, num_leaves, max_depth, feat_frac, bag_frac, min_data) {
  lgb_params <- list(
    objective        = "binary",
    metric           = "binary_logloss",
    learning_rate    = lr,
    num_leaves       = num_leaves,
    max_depth        = max_depth,
    feature_fraction = feat_frac,
    bagging_fraction = bag_frac,
    bagging_freq     = 1,
    min_data_in_leaf = min_data,
    verbosity        = -1L
  )
  
  # L1
  smote_l1 <- smote_binary_data(train_scaled, layer_feat_names, ifelse(train_scaled$target_col == "1", 1, 0), seed = config$training$random_state)
  dtr_l1   <- lgb.Dataset(smote_l1$X, label = smote_l1$y)
  dvl_l1   <- lgb.Dataset(as.matrix(val_scaled[, layer_feat_names]), label = ifelse(val_scaled$target_col == "1", 1, 0))
  l1_m     <- lgb.train(params = lgb_params, data = dtr_l1, nrounds = 100, valids = list(val = dvl_l1), early_stopping_rounds = 10, verbose = -1L)
  
  # L2
  tr_l2 <- train_scaled %>% filter(target_col != "1")
  vl_l2 <- val_scaled   %>% filter(target_col != "1")
  smote_l2 <- smote_binary_data(tr_l2, layer_feat_names, ifelse(tr_l2$target_col %in% c("2", "3"), 1, 0), seed = config$training$random_state)
  dtr_l2   <- lgb.Dataset(smote_l2$X, label = smote_l2$y)
  dvl_l2   <- lgb.Dataset(as.matrix(vl_l2[, layer_feat_names]), label = ifelse(vl_l2$target_col %in% c("2", "3"), 1, 0))
  l2_m     <- lgb.train(params = lgb_params, data = dtr_l2, nrounds = 100, valids = list(val = dvl_l2), early_stopping_rounds = 10, verbose = -1L)
  
  # L3A
  tr_l3a <- train_scaled %>% filter(target_col %in% c("2", "3"))
  vl_l3a <- val_scaled   %>% filter(target_col %in% c("2", "3"))
  smote_l3a <- smote_binary_data(tr_l3a, layer_feat_names, ifelse(tr_l3a$target_col == "2", 1, 0), seed = config$training$random_state)
  dtr_l3a   <- lgb.Dataset(smote_l3a$X, label = smote_l3a$y)
  dvl_l3a   <- lgb.Dataset(as.matrix(vl_l3a[, layer_feat_names]), label = ifelse(vl_l3a$target_col == "2", 1, 0))
  l3a_m     <- lgb.train(params = lgb_params, data = dtr_l3a, nrounds = 100, valids = list(val = dvl_l3a), early_stopping_rounds = 10, verbose = -1L)
  
  # L3B
  tr_l3b <- train_scaled %>% filter(target_col %in% c("4", "5"))
  vl_l3b <- val_scaled   %>% filter(target_col %in% c("4", "5"))
  smote_l3b <- smote_binary_data(tr_l3b, layer_feat_names, ifelse(tr_l3b$target_col == "4", 1, 0), seed = config$training$random_state)
  dtr_l3b   <- lgb.Dataset(smote_l3b$X, label = smote_l3b$y)
  dvl_l3b   <- lgb.Dataset(as.matrix(vl_l3b[, layer_feat_names]), label = ifelse(vl_l3b$target_col == "4", 1, 0))
  l3b_m     <- lgb.train(params = lgb_params, data = dtr_l3b, nrounds = 100, valids = list(val = dvl_l3b), early_stopping_rounds = 10, verbose = -1L)
  
  p1_val  <- predict(l1_m,  as.matrix(val_scaled[, layer_feat_names]))
  p2_val  <- predict(l2_m,  as.matrix(val_scaled[, layer_feat_names]))
  p3a_val <- predict(l3a_m, as.matrix(val_scaled[, layer_feat_names]))
  p3b_val <- predict(l3b_m, as.matrix(val_scaled[, layer_feat_names]))
  
  probs_val <- matrix(0, nrow = nrow(val_df), ncol = 5)
  probs_val[, 1] <- p1_val
  probs_val[, 2] <- (1 - p1_val) * p2_val * p3a_val
  probs_val[, 3] <- (1 - p1_val) * p2_val * (1 - p3a_val)
  probs_val[, 4] <- (1 - p1_val) * (1 - p2_val) * p3b_val
  probs_val[, 5] <- (1 - p1_val) * (1 - p2_val) * (1 - p3b_val)
  
  preds_val <- apply(probs_val, 1, which.max)
  act_val   <- as.numeric(as.character(val_df$target_col))
  
  bal_accs <- numeric(5)
  for (k in 1:5) {
    tp <- sum(preds_val == k & act_val == k)
    fn <- sum(preds_val != k & act_val == k)
    fp <- sum(preds_val == k & act_val != k)
    tn <- sum(preds_val != k & act_val != k)
    sens <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
    spec <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
    bal_accs[k] <- (sens + spec) / 2
  }
  return(mean(bal_accs))
}
''')
def objective_stage1(trial):
    lr        = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    num_leaves= trial.suggest_int('num_leaves', 15, 63)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    feat_frac = trial.suggest_float('feature_fraction', 0.5, 1.0)
    bag_frac  = trial.suggest_float('bagging_fraction', 0.5, 1.0)
    min_data  = trial.suggest_int('min_data_in_leaf', 10, 100)
    
    score = r['evaluate_lgb_hyperparams'](lr, num_leaves, max_depth, feat_frac, bag_frac, min_data)[0]
    return score
print("=== Starting Stage 1 Optuna Optimization (100 Trials for LightGBM Hyperparameters) ===")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_stage1 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_stage1.optimize(objective_stage1, n_trials=100)
best_lgb_params = study_stage1.best_params
print(f"  Stage 1 Best Validation Macro Balanced Accuracy: {study_stage1.best_value:.4f}")
print("  Optimal LightGBM Hyperparameters:")
for k, v in best_lgb_params.items():
    print(f"    {k}: {v}")

In [ ]:
%%R -i best_lgb_params
# ---------------------------------------------------------
# Step 3: Train 4 Sub-Models with Stage 1 Optimal LightGBM Hyperparameters (Silenced Logs)
# ---------------------------------------------------------
opt_lgb_params <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = as.numeric(best_lgb_params[["learning_rate"]]),
  num_leaves       = as.integer(best_lgb_params[["num_leaves"]]),
  max_depth        = as.integer(best_lgb_params[["max_depth"]]),
  feature_fraction = as.numeric(best_lgb_params[["feature_fraction"]]),
  bagging_fraction = as.numeric(best_lgb_params[["bagging_fraction"]]),
  bagging_freq     = 1,
  min_data_in_leaf = as.integer(best_lgb_params[["min_data_in_leaf"]]),
  verbosity        = -1L
)
# L1
smote_l1 <- smote_binary_data(train_scaled, layer_feat_names, ifelse(train_scaled$target_col == "1", 1, 0), seed = config$training$random_state)
dtr_l1   <- lgb.Dataset(smote_l1$X, label = smote_l1$y)
dvl_l1   <- lgb.Dataset(as.matrix(val_scaled[, layer_feat_names]), label = ifelse(val_scaled$target_col == "1", 1, 0))
l1_model <- lgb.train(params = opt_lgb_params, data = dtr_l1, nrounds = 100, valids = list(val = dvl_l1), early_stopping_rounds = 10, verbose = -1L)
# L2
tr_l2 <- train_scaled %>% filter(target_col != "1")
vl_l2 <- val_scaled   %>% filter(target_col != "1")
smote_l2 <- smote_binary_data(tr_l2, layer_feat_names, ifelse(tr_l2$target_col %in% c("2", "3"), 1, 0), seed = config$training$random_state)
dtr_l2   <- lgb.Dataset(smote_l2$X, label = smote_l2$y)
dvl_l2   <- lgb.Dataset(as.matrix(vl_l2[, layer_feat_names]), label = ifelse(vl_l2$target_col %in% c("2", "3"), 1, 0))
l2_model <- lgb.train(params = opt_lgb_params, data = dtr_l2, nrounds = 100, valids = list(val = dvl_l2), early_stopping_rounds = 10, verbose = -1L)
# L3A
tr_l3a <- train_scaled %>% filter(target_col %in% c("2", "3"))
vl_l3a <- val_scaled   %>% filter(target_col %in% c("2", "3"))
smote_l3a <- smote_binary_data(tr_l3a, layer_feat_names, ifelse(tr_l3a$target_col == "2", 1, 0), seed = config$training$random_state)
dtr_l3a   <- lgb.Dataset(smote_l3a$X, label = smote_l3a$y)
dvl_l3a   <- lgb.Dataset(as.matrix(vl_l3a[, layer_feat_names]), label = ifelse(vl_l3a$target_col == "2", 1, 0))
l3a_model <- lgb.train(params = opt_lgb_params, data = dtr_l3a, nrounds = 100, valids = list(val = dvl_l3a), early_stopping_rounds = 10, verbose = -1L)
# L3B
tr_l3b <- train_scaled %>% filter(target_col %in% c("4", "5"))
vl_l3b <- val_scaled   %>% filter(target_col %in% c("4", "5"))
smote_l3b <- smote_binary_data(tr_l3b, layer_feat_names, ifelse(tr_l3b$target_col == "4", 1, 0), seed = config$training$random_state)
dtr_l3b   <- lgb.Dataset(smote_l3b$X, label = smote_l3b$y)
dvl_l3b   <- lgb.Dataset(as.matrix(vl_l3b[, layer_feat_names]), label = ifelse(vl_l3b$target_col == "4", 1, 0))
l3b_model <- lgb.train(params = opt_lgb_params, data = dtr_l3b, nrounds = 100, valids = list(val = dvl_l3b), early_stopping_rounds = 10, verbose = -1L)
p1_val  <- predict(l1_model,  as.matrix(val_scaled[, layer_feat_names]))
p2_val  <- predict(l2_model,  as.matrix(val_scaled[, layer_feat_names]))
p3a_val <- predict(l3a_model, as.matrix(val_scaled[, layer_feat_names]))
p3b_val <- predict(l3b_model, as.matrix(val_scaled[, layer_feat_names]))
probs_val <- matrix(0, nrow = nrow(val_df), ncol = 5)
probs_val[, 1] <- p1_val
probs_val[, 2] <- (1 - p1_val) * p2_val * p3a_val
probs_val[, 3] <- (1 - p1_val) * p2_val * (1 - p3a_val)
probs_val[, 4] <- (1 - p1_val) * (1 - p2_val) * p3b_val
probs_val[, 5] <- (1 - p1_val) * (1 - p2_val) * (1 - p3b_val)
y_val_act <- as.numeric(as.character(val_df$target_col))
p1_test  <- predict(l1_model,  as.matrix(test_scaled[, layer_feat_names]))
p2_test  <- predict(l2_model,  as.matrix(test_scaled[, layer_feat_names]))
p3a_test <- predict(l3a_model, as.matrix(test_scaled[, layer_feat_names]))
p3b_test <- predict(l3b_model, as.matrix(test_scaled[, layer_feat_names]))
probs_test <- matrix(0, nrow = nrow(test_df), ncol = 5)
probs_test[, 1] <- p1_test
probs_test[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_test[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_test[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_test[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
y_test_act <- as.numeric(as.character(test_df$target_col))
# Save Model Artifacts
saveRDS(list(model = l1_model,  scaler = scaler, is_l1_lgb  = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
saveRDS(list(model = l2_model,  scaler = scaler, is_lgb_l2  = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
saveRDS(list(model = l3a_model, scaler = scaler, is_lgb_l3a = TRUE), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = l3b_model, scaler = scaler, is_lgb_l3b = TRUE), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
cat("Sub-Models Trained with Stage 1 Optimal LightGBM Hyperparameters (Logs Silenced) & Artifacts Saved!\n")

In [ ]:
# ---------------------------------------------------------
# Step 4: STAGE 2 OPTUNA - Tune Class Multiplier Weights
# ---------------------------------------------------------
from sklearn.metrics import roc_auc_score
P_val = np.array(r('probs_val'))
y_val = np.array(r('y_val_act'))
P_test = np.array(r('probs_test'))
y_test = np.array(r('y_test_act'))
def compute_macro_balanced_acc(y_true, y_pred):
    classes = [1, 2, 3, 4, 5]
    bal_accs = []
    for cls in classes:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        tn = np.sum((y_true != cls) & (y_pred != cls))
        
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((sens + spec) / 2.0)
    return np.mean(bal_accs)
def objective_stage2(trial):
    w1 = trial.suggest_float('w1', 0.1, 20.0, log=True)
    w2 = trial.suggest_float('w2', 0.1, 20.0, log=True)
    w3 = trial.suggest_float('w3', 0.1, 20.0, log=True)
    w4 = trial.suggest_float('w4', 0.1, 20.0, log=True)
    w5 = trial.suggest_float('w5', 0.1, 20.0, log=True)
    
    weights = np.array([w1, w2, w3, w4, w5])
    weighted_probs = P_val * weights
    preds = np.argmax(weighted_probs, axis=1) + 1
    return compute_macro_balanced_acc(y_val, preds)
print("=== Starting Stage 2 Optuna Optimization (200 Trials for Class Multiplier Weights) ===")
study_stage2 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_stage2.optimize(objective_stage2, n_trials=200)
best_weights = study_stage2.best_params
best_val_score_stage2 = study_stage2.best_value
print(f"  Stage 2 Best Validation Macro Balanced Accuracy: {best_val_score_stage2:.4f}")
print("  Optimal Class Multiplier Weights:")
for k, v in best_weights.items():
    print(f"    {k}: {v:.4f}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Holdout Test Set Evaluation & Artifact Reporting
# ---------------------------------------------------------
base_preds_test = np.argmax(P_test, axis=1) + 1
opt_weights = np.array([best_weights['w1'], best_weights['w2'], best_weights['w3'], best_weights['w4'], best_weights['w5']])
opt_probs_test = P_test * opt_weights
opt_preds_test = np.argmax(opt_probs_test, axis=1) + 1
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        
        try:
            auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception:
            auc = 0.0
            
        recalls.append(rec)
        specs.append(spec)
        bal_accs.append(bal)
        aucs.append(auc)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
        
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    
    return pd.DataFrame(rows)
base_df = get_per_class_breakdown(y_test, base_preds_test, P_test, 'Stage1_Tuned_Unweighted_Soft_Model')
opt_df  = get_per_class_breakdown(y_test, opt_preds_test, opt_probs_test, 'Stage2_Tuned_Weighted_Soft_Model')
full_report_df = pd.concat([base_df, opt_df], ignore_index=True)
print("========================================================================================")
print("   2-STAGE OPTUNA HOLDOUT TEST REPORT (38 FEATS - ESI 1..5 RECALL, SPECIFICITY, BAL ACC, ROC-AUC)")
print("========================================================================================")
print(full_report_df.to_string(index=False))
print("========================================================================================\n")
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(deploy_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)
with open(os.path.join(deploy_dir, 'soft_pipeline_best_lgb_params.json'), 'w') as f:
    json.dump(best_lgb_params, f, indent=2)
opt_weights_dict = {
  'features_count': 38,
  'resampling': 'SMOTE',
  'weights': [float(best_weights[f'w{i}']) for i in range(1, 6)],
  'val_macro_balanced_accuracy_stage1': float(study_stage1.best_value),
  'val_macro_balanced_accuracy_stage2': float(best_val_score_stage2),
  'test_macro_balanced_accuracy': float(opt_df[opt_df['Class']=='Macro_Average']['Balanced_Accuracy'].values[0])
}
with open(os.path.join(deploy_dir, 'soft_pipeline_optimized_weights_smote.json'), 'w') as f:
    json.dump(opt_weights_dict, f, indent=2)
full_report_df.to_csv(os.path.join(reports_dir, 'optimized_soft_pipeline_per_class_report_smote.csv'), index=False)
print("Artifacts Saved:")
print(f"  - {os.path.join(deploy_dir, 'soft_pipeline_best_lgb_params.json')}")
print(f"  - {os.path.join(deploy_dir, 'soft_pipeline_optimized_weights_smote.json')}")
print(f"  - {os.path.join(reports_dir, 'optimized_soft_pipeline_per_class_report_smote.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Plot Per-Class Metrics Bar Chart (ESI 1..5 Recall, Specificity, ROC-AUC)
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
esi_classes_df = full_report_df[full_report_df['Class'] != 'Macro_Average']
df_melted = pd.melt(esi_classes_df, id_vars=['Pipeline', 'Class'], value_vars=['Recall', 'Specificity', 'ROC_AUC'], var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=df_melted, x='Class', y='Score', hue='Pipeline', palette=['#1f77b4', '#2ca02c'])
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=8, xytext=(0, 2),
                    textcoords='offset points')
plt.title('2-Stage Optuna Optimization (38 Features): Stage 1 Hyperparameter Tuning vs Stage 2 Multiplier Weights', fontsize=12, fontweight='bold', pad=15)
plt.ylim(0, 1.15)
plt.ylabel('Score', fontsize=11)
plt.xlabel('ESI Triage Level', fontsize=11)
plt.legend(title='Pipeline Stage', loc='upper right')
plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, 'optimized_soft_pipeline_per_class_comparison_smote.png'), dpi=300)
plt.show()
print(f"Per-Class Plot saved to {os.path.join(plots_dir, 'optimized_soft_pipeline_per_class_comparison_smote.png')}")